# 🎯 Arabic Pronunciation Assessment – Production Backend

**State-of-the-art Pipeline** (fest konfiguriert, keine Alternativen):

| Stufe | Tool | Zweck |
|---|---|---|
| VAD / Trim | **Silero VAD 5** | Stille am Anfang/Ende entfernen |
| ASR | **wav2vec 2.0 XLSR-53 Arabic** | Beste offene Arabisch-Erkennung |
| Alignment | **`torchaudio.functional.forced_align`** | CUDA-beschleunigte CTC-Alignment |
| Scoring | **GOP** (mittlere Log-Prob pro Buchstabe) | 0 – 100 pro Buchstabe |
| Serving | **FastAPI + Uvicorn**, Pydantic-Validierung | Getypter Endpoint mit Limits |
| Tunnel | **Cloudflared Quick Tunnel** | Öffentliche URL ohne Login |

**Ausführung in Colab:**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. Letzte Zelle druckt die URL für `index.html`.

In [ ]:
%pip install -q -U transformers pydub nest_asyncio python-multipart
%pip install -q -U fastapi 'uvicorn[standard]'
%pip install -q -U silero-vad

# scipy bewusst NICHT hier upgraden: die Colab-Basis liefert scipy vorinstalliert.
# Ein "pip install -U scipy" waehrend einer laufenden Session verursacht in Colab
# einen Cython-ABI-Mismatch (siehe: scipy._cyutility / __Pyx__Import ImportError),
# weil numpy/scipy-Extensions bereits im Speicher liegen.

!apt-get -qq install -y ffmpeg > /dev/null
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import importlib
for mod in ("transformers", "pydub", "fastapi", "uvicorn", "silero_vad", "torch", "torchaudio", "scipy"):
    m = importlib.import_module(mod)
    print(f"  ✅ {mod:14s} {getattr(m, '__version__', '?')}")
print("✅ Alle Abhängigkeiten importierbar.")

In [ ]:
import io, os, re, time, subprocess, threading, urllib.request, unicodedata
from typing import List, Dict, Any

import numpy as np
import torch
import torchaudio.functional as AF
from pydub import AudioSegment
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from silero_vad import load_silero_vad, get_speech_timestamps

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SR = 16000
print(f"✅ Device: {device}, torch {torch.__version__}")

In [ ]:
ASR_MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"

# fp16 nur auf GPU: ~2x schnellere Inferenz, kein Qualitaetsverlust bei CTC.
USE_FP16 = device.type == "cuda"
DTYPE    = torch.float16 if USE_FP16 else torch.float32

print("Lade ASR-Modell …")
asr_processor = Wav2Vec2Processor.from_pretrained(ASR_MODEL_ID)
asr_model     = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL_ID).to(device=device, dtype=DTYPE).eval()
ASR_VOCAB    = asr_processor.tokenizer.get_vocab()
ASR_BLANK_ID = asr_model.config.pad_token_id
print(f"  ✅ wav2vec2 XLSR-53 Arabic  ({len(ASR_VOCAB)} tokens, blank={ASR_BLANK_ID}, dtype={DTYPE})")

print("Lade Silero VAD …")
vad_model = load_silero_vad()
print("  ✅ Silero VAD 5")

# Warm-up mit realistischer 2s-Laenge, damit spaeter kein CUDA-JIT die erste Anfrage bremst.
with torch.inference_mode():
    _ = asr_model(torch.zeros(1, 2 * SR, device=device, dtype=DTYPE)).logits
print("✅ Modelle geladen und aufgewärmt.")

In [ ]:
from scipy.signal import butter, sosfiltfilt

# Signalkette vor dem ASR (alle Schritte sind linear, phasentreu oder nicht-neuronal
# -> beruehren die spektrale Signatur arabischer Gutturale/Emphatika/Frikative nicht):
#   decode  -> HP80Hz  -> RMS-Norm -> gentle_trim (VAD) -> Kontext-Pad
_HPF_SOS = butter(2, 80.0, btype="highpass", fs=SR, output="sos")

def decode_audio(raw: bytes) -> np.ndarray:
    """Beliebiges Audioformat -> 16 kHz mono float32."""
    seg = AudioSegment.from_file(io.BytesIO(raw))
    seg = seg.set_frame_rate(SR).set_channels(1).set_sample_width(2)
    return np.asarray(seg.get_array_of_samples(), dtype=np.float32) / 32768.0

def _highpass(audio: np.ndarray) -> np.ndarray:
    """Nullphasiger Butterworth @ 80 Hz: entfernt DC-Offset, Handling-Rumpeln,
    Netzbrummen (50/60 Hz). Liegt unterhalb jeder Sprachformantenergie."""
    return sosfiltfilt(_HPF_SOS, audio).astype(np.float32)

def _normalize_level(audio: np.ndarray, target_dbfs: float = -20.0) -> np.ndarray:
    """RMS-Normalisierung auf konsistentes Pegel -> Silero-VAD-Schwelle wird reproduzierbar,
    und wav2vec2s eingebautes do_normalize=True bekommt ein saubereres Zero-Mean/Unit-Var-Ziel."""
    rms = float(np.sqrt(np.mean(audio ** 2)))
    if rms < 1e-6:
        return audio
    gain = 10.0 ** ((target_dbfs - 20.0 * np.log10(rms)) / 20.0)
    out  = audio * gain
    peak = float(np.max(np.abs(out)))
    if peak > 0.99:
        out = out / peak * 0.99
    return out.astype(np.float32)

def gentle_trim(audio: np.ndarray, pad_ms: int = 120) -> np.ndarray:
    """Nur führende/nachlaufende lange Stille entfernen. Zwischenpausen bleiben."""
    segs = get_speech_timestamps(torch.from_numpy(audio), vad_model,
                                 sampling_rate=SR, threshold=0.35)
    if not segs:
        return audio
    pad = int(pad_ms * SR / 1000)
    start = max(0, segs[0]["start"] - pad)
    end   = min(len(audio), segs[-1]["end"] + pad)
    return audio[start:end]

def _pad_context(audio: np.ndarray, ms: int = 250) -> np.ndarray:
    """Wav2vec2-Transformer sieht pro Frame ein bidirektionales Kontextfenster (~200 ms).
    Kurze Woerter (2-3 Buchstaben) verlieren sonst am Anfang/Ende Kontextframes und werden
    systematisch schlechter erkannt. Silence-Padding kostet keine Latenz und keine Genauigkeit."""
    pad = np.zeros(int(ms * SR / 1000), dtype=np.float32)
    return np.concatenate([pad, audio, pad])

def preprocess(raw: bytes) -> np.ndarray:
    audio = decode_audio(raw)
    audio = _highpass(audio)
    audio = _normalize_level(audio)
    audio = gentle_trim(audio)
    audio = _pad_context(audio)
    return audio

In [ ]:
# Nur klassisches Tashkeel entfernen. Hamza-Formen (أ إ آ ؤ ئ) bleiben als eigene Buchstaben erhalten.
_TASHKEEL = set("ًٌٍَُِّْٰ")

def strip_diacritics(text: str) -> str:
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if c not in _TASHKEEL))

# Positionsabhaengige Aequivalenzen (Anfang/Ende) fuer Posterior-Bewertung.
_START_EQUIV = {ch: "اأإآ" for ch in "اأإآ"}
_END_EQUIV   = {"ة": "ةه", "ه": "هة",
                "ى": "ىيا", "ي": "يى"}

# Linguistisch belegte Verwechslungen fuer den LLR-Test.
# Quellen: Al-Ani (1970) "Arabic Phonology"; Newman (2013);
# Standard-DaF/L2-Arabisch-Fehlerkataloge; Kinder-L1-Erwerbsstudien.
_CONFUSABLES: Dict[str, str] = {
    "ت": "طثد",
    "ث": "تسذف",
    "ح": "هخع",
    "خ": "حغك",
    "د": "تضذ",
    "ذ": "دزثظ",
    "ر": "لغ",
    "ز": "ذسظ",
    "س": "صثزش",
    "ش": "سج",
    "ص": "سض",
    "ض": "دظص",
    "ط": "تضد",
    "ظ": "زذض",
    "ع": "ءأاه",
    "غ": "خقر",
    "ق": "كغخ",
    "ك": "قخج",
    "ل": "ر",
    "ه": "حة",
    "ء": "ع",
    "ج": "شك",
}

def _equiv_ids(ch: str, pos: int, total: int) -> List[int]:
    if pos == 0 and ch in _START_EQUIV:
        alts = _START_EQUIV[ch]
    elif pos == total - 1 and ch in _END_EQUIV:
        alts = _END_EQUIV[ch]
    else:
        alts = ch
    ids = [ASR_VOCAB[c] for c in alts if c in ASR_VOCAB]
    return ids or [ASR_VOCAB[ch]]

def _confuse_ids(ch: str) -> List[int]:
    alts = _CONFUSABLES.get(ch, "")
    return [ASR_VOCAB[c] for c in alts if c in ASR_VOCAB]

# Umkehr-Map: Token-ID -> Buchstabe, fuer error_hint.
_ID_TO_CHAR = {tid: c for c, tid in ASR_VOCAB.items()}

def encode_target(word: str) -> List[int]:
    ids: List[int] = []
    for ch in word:
        tid = ASR_VOCAB.get(ch)
        if tid is None:
            raise ValueError(f"Zeichen {ch!r} nicht im ASR-Vokabular.")
        ids.append(tid)
    return ids

@torch.inference_mode()
def run_asr(audio: np.ndarray):
    inputs = asr_processor(audio, sampling_rate=SR, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device=device, dtype=DTYPE)
    logits = asr_model(input_values).logits
    # log_softmax stabil in fp32, torchaudio.forced_align verlangt float32 CPU.
    log_probs = torch.log_softmax(logits.float(), dim=-1).cpu()
    transcription = asr_processor.batch_decode(log_probs.argmax(dim=-1))[0]
    return log_probs, transcription

def _runs_of_non_blank(tokens: List[int]) -> List[List[int]]:
    runs: List[List[int]] = []
    current: List[int] = []
    last: int = -1
    for t, tok in enumerate(tokens):
        if tok == ASR_BLANK_ID:
            if current: runs.append(current); current = []
            last = -1
        elif tok != last:
            if current: runs.append(current)
            current = [t]; last = tok
        else:
            current.append(t)
    if current: runs.append(current)
    return runs

# Kalibrierungskonstante: LLR=0 -> 50, LLR=+1 -> ~88, LLR=-1 -> ~12.
_LLR_K = 2.0

def _sigmoid(x: float) -> float:
    return 1.0 / (1.0 + float(np.exp(-x)))

def gop_score(log_probs: torch.Tensor, target_word: str) -> List[Dict[str, Any]]:
    target_ids = encode_target(target_word)
    if not target_ids:
        return []
    if log_probs.shape[1] < len(target_ids):
        raise ValueError("Aufnahme zu kurz für dieses Wort.")
    targets = torch.tensor([target_ids], dtype=torch.int32)
    aligned, _ = AF.forced_align(log_probs, targets, blank=ASR_BLANK_ID)
    runs = _runs_of_non_blank(aligned[0].tolist())
    total_len = len(target_word)
    results: List[Dict[str, Any]] = []
    for i, ch in enumerate(target_word):
        if i >= len(runs):
            results.append({"label": ch, "score": 0.0, "confidence": 0.0,
                            "llr": -5.0, "error_hint": None})
            continue

        frames   = runs[i]
        lp_frame = log_probs[0, frames]  # [F, V]

        # 1) Posterior-Score (klassisches GOP, positionsbewusst).
        equiv_ids  = _equiv_ids(ch, i, total_len)
        target_lp  = lp_frame[:, equiv_ids].max(dim=-1).values.mean().item()
        post_score = float(np.clip((target_lp + 3.0) / 3.0 * 100, 0, 100))
        conf       = float(np.exp(target_lp))

        # 2) LLR gegen dokumentierte Verwechslungen (Anti-Modell).
        confuse_ids = _confuse_ids(ch)
        if confuse_ids:
            per_frame_conf = lp_frame[:, confuse_ids]
            best_conf_lp   = per_frame_conf.max(dim=-1).values.mean().item()
            llr            = target_lp - best_conf_lp
            llr_score      = _sigmoid(_LLR_K * llr) * 100.0
            # Nur melden wenn Verwechslung staerker als Ziel.
            if llr < 0:
                best_col   = int(per_frame_conf.mean(dim=0).argmax().item())
                hint_id    = confuse_ids[best_col]
                error_hint = _ID_TO_CHAR.get(hint_id)
            else:
                error_hint = None
        else:
            llr, llr_score, error_hint = 5.0, 100.0, None

        # 3) Kombination: 40 % Posterior + 60 % LLR (LLR ist informativer).
        final = 0.4 * post_score + 0.6 * llr_score
        results.append({
            "label": ch,
            "score": float(np.clip(final, 0, 100)),
            "confidence": conf,
            "llr": float(llr),
            "error_hint": error_hint,
        })
    return results

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import asyncio, json

MAX_AUDIO_BYTES = 3 * 1024 * 1024
MIN_SAMPLES     = int(0.15 * SR)

# Cloudflared Quick Tunnel killt WS-Verbindungen nach ~60-100s Inaktivitaet.
# Wir senden alle 20s einen App-Level-Ping, damit die Verbindung fuer die
# ganze Kind-Session offen bleibt (sonst Reconnect-Cost pro Wort ~500ms).
WS_KEEPALIVE_SEC = 20

class Unit(BaseModel):
    label: str
    score: float
    confidence: float
    llr: Optional[float] = None
    error_hint: Optional[str] = None

class AssessResponse(BaseModel):
    target: str
    transcription: str
    units: List[Unit]
    total: float
    duration_ms: int

app = FastAPI(title="Arabic Pronunciation API", version="1.3.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

def _score_word(raw: bytes, target: str) -> Dict[str, Any]:
    """Synchrone Bewertungs-Pipeline. Wird sowohl vom HTTP- als auch vom WS-Endpoint aufgerufen,
    damit die Bewertungsqualitaet identisch bleibt."""
    if len(raw) > MAX_AUDIO_BYTES:
        raise HTTPException(413, f"Audio > {MAX_AUDIO_BYTES // 1024} KB.")
    if not raw:
        raise HTTPException(400, "Leere Audiodatei.")
    try:
        wav = preprocess(raw)
    except Exception as e:
        raise HTTPException(400, f"Audio ungültig: {e}")
    if wav.size < MIN_SAMPLES:
        raise HTTPException(400, "Aufnahme zu kurz.")
    target_clean = strip_diacritics(target)
    log_probs, transcription = run_asr(wav)
    units = gop_score(log_probs, target_clean)
    total = float(np.mean([u["score"] for u in units])) if units else 0.0
    return {
        "target": target_clean,
        "transcription": transcription,
        "units": units,
        "total": total,
    }

@app.get("/health")
def health():
    return {"status": "ok", "device": str(device),
            "asr_model": ASR_MODEL_ID, "vad": "Silero VAD 5",
            "fp16": USE_FP16,
            "endpoints": ["/assess (HTTP)", "/stream (WebSocket)"]}

@app.post("/assess", response_model=AssessResponse)
def assess(audio: UploadFile = File(...), target: str = Form(...)):
    target = target.strip()
    if not target:
        raise HTTPException(400, "Zielwort fehlt.")
    t0 = time.perf_counter()
    raw = audio.file.read(MAX_AUDIO_BYTES + 1)
    try:
        result = _score_word(raw, target)
    except ValueError as e:
        raise HTTPException(400, str(e))
    result["duration_ms"] = int((time.perf_counter() - t0) * 1000)
    return AssessResponse(units=[Unit(**u) for u in result["units"]], **{
        k: v for k, v in result.items() if k != "units"
    })

@app.websocket("/stream")
async def stream_ws(ws: WebSocket):
    """Persistente Session pro Kind. Protokoll pro Wort:
       1. Client -> Server: Text-Frame {"target": "\u0643\u062a\u0627\u0628"}
       2. Client -> Server: Binaer-Frame mit kodiertem Audio (m4a/wav/webm)
       3. Server -> Client: JSON-Frame mit AssessResponse ODER {"error": "..."}
       Parallel: Server sendet alle WS_KEEPALIVE_SEC Sekunden {"ping": true}
       damit Cloudflared die Verbindung nicht als idle killt."""
    await ws.accept()

    async def keepalive():
        try:
            while True:
                await asyncio.sleep(WS_KEEPALIVE_SEC)
                await ws.send_json({"ping": True})
        except Exception:
            return

    ka_task = asyncio.create_task(keepalive())

    try:
        while True:
            ctrl = json.loads(await ws.receive_text())
            target = str(ctrl.get("target", "")).strip()
            if not target:
                await ws.send_json({"error": "Zielwort fehlt."})
                continue
            msg = await ws.receive()
            if "bytes" not in msg or msg["bytes"] is None:
                await ws.send_json({"error": "Erwartete Binaerdaten (Audio)."})
                continue
            raw: bytes = msg["bytes"]
            t0 = time.perf_counter()
            try:
                result = await asyncio.to_thread(_score_word, raw, target)
            except HTTPException as e:
                await ws.send_json({"error": e.detail})
                continue
            except ValueError as e:
                await ws.send_json({"error": str(e)})
                continue
            except Exception as e:
                await ws.send_json({"error": f"Serverfehler: {e}"})
                continue
            result["duration_ms"] = int((time.perf_counter() - t0) * 1000)
            await ws.send_json(result)
    except WebSocketDisconnect:
        return
    except Exception as e:
        try: await ws.send_json({"error": f"Serverfehler: {e}"})
        except Exception: pass
    finally:
        ka_task.cancel()

print("✅ API definiert:  GET /health   POST /assess   WS /stream  (Keep-Alive alle 20s)")

In [ ]:
import uvicorn, nest_asyncio
nest_asyncio.apply()

PORT   = 8000
CF_LOG = "/tmp/cf.log"

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT,
                               log_level="warning", access_log=False),
    daemon=True,
).start()

for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1).read()
        print(f"✅ FastAPI läuft auf Port {PORT}")
        break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError("FastAPI-Start fehlgeschlagen.")

if os.path.exists(CF_LOG):
    os.remove(CF_LOG)
subprocess.Popen(f"cloudflared tunnel --url http://localhost:{PORT} > {CF_LOG} 2>&1 &", shell=True)

public_url = None
for _ in range(60):
    time.sleep(1)
    if os.path.exists(CF_LOG):
        m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", open(CF_LOG).read())
        if m:
            public_url = m.group(0)
            break

if not public_url:
    raise RuntimeError("Keine Tunnel-URL. Log:\n" + open(CF_LOG).read()[-800:])

print("\n" + "=" * 68)
print(f"🌍 Backend-URL für index.html:  {public_url}")
print("=" * 68)
print(f"Health-Check:  {public_url}/health")